<a href="https://colab.research.google.com/github/vcasadei/Redes-Neurais-CESAR-School/blob/2026.01-A/4%20-%20Redes%20Neurais/4.1_funcoes_de_ativacao.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir no Colab"/></a>

# 4.1 — Funções de ativação

Nos notebooks 3.x, a sigmoide e a softmax apareceram na saída do modelo, para transformar a saída linear $z = \boldsymbol{w}^T\boldsymbol{x} + b$ em probabilidade. Numa rede neural, uma função de ativação $f$ também é aplicada **entre as camadas**: cada camada calcula $\boldsymbol{a} = f(W\boldsymbol{x} + \boldsymbol{b})$.

A ativação precisa ser não linear. Sem ela, duas camadas lineares seguidas, $W_2(W_1\boldsymbol{x} + \boldsymbol{b}_1) + \boldsymbol{b}_2 = (W_2W_1)\boldsymbol{x} + (W_2\boldsymbol{b}_1 + \boldsymbol{b}_2)$, equivalem a uma única camada linear, e empilhar camadas não aumenta o que o modelo consegue representar.

Este notebook apresenta a equação e o gráfico de algumas ativações, primeiro em NumPy e depois com as classes do `torch.nn`. No 4.2, elas serão usadas numa rede com camadas escondidas.

## Objetivos

- Conhecer a equação, o intervalo de saída e o gráfico da sigmoide, tanh, softsign, ReLU e ELU.
- Usar as ativações do `torch.nn` sobre tensores.
- Calcular a derivada de cada ativação com o autograd e ver onde ela satura (derivada perto de zero).

## Importação das bibliotecas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

## Ativações implementadas em NumPy

Em todos os gráficos, $x$ é a entrada do neurônio (a soma ponderada $z$) e o eixo vertical é a saída da ativação.

### Sigmoide (também chamada de função logística)

A sigmoide tem a forma de um "S" e leva qualquer número real para o intervalo $(0, 1)$:

$$\sigma(x) = \frac{1}{1+e^{-x}}$$

Por isso ela é usada para converter um valor em probabilidade, como na regressão logística. Sua derivada é $\sigma'(x) = \sigma(x)\,(1-\sigma(x))$, que vale no máximo $0{,}25$ (em $x = 0$) e tende a zero quando $|x|$ é grande.

In [ ]:
x = np.linspace(-8, 8, 100)

s = 1. / (1. + np.exp(-x))

plt.plot(x, s)
plt.title('Sigmoide')
plt.axhline(0.5, ls='dashed')
plt.axvline(0);

### ReLU (Rectified Linear Unit)

A ReLU, também chamada de função rampa, zera os valores negativos:

$$f(x) = \max(0, x) = \begin{cases} x & \text{se } x \ge 0 \\ 0 & \text{caso contrário} \end{cases}$$

Apesar de simples, ela funciona muito bem no treinamento de redes profundas e é a escolha padrão nas camadas escondidas. Glorot, Bordes e Bengio (2011) apontam as seguintes vantagens:

- inspiração biológica: neurônios reais ficam inativos para boa parte das entradas;
- ativação esparsa: com pesos iniciados ao acaso, cerca de metade dos neurônios tem saída zero;
- derivada igual a 1 para $x > 0$: o gradiente não diminui ao atravessar a ativação, ao contrário da sigmoide e da tanh;
- custo computacional baixo (uma comparação);
- invariância à escala: $\max(0, \alpha x) = \alpha \max(0, x)$ para $\alpha \ge 0$.

A desvantagem é que a derivada é zero para $x < 0$: um neurônio cuja entrada fica sempre negativa para de aprender.

Referência: [Glorot, X.; Bordes, A.; Bengio, Y. (2011). Deep Sparse Rectifier Neural Networks. AISTATS, PMLR 15, pp. 315–323.](https://proceedings.mlr.press/v15/glorot11a/glorot11a.pdf)

In [ ]:
x = np.linspace(-8, 8, 100)

s = np.maximum(x, 0)

plt.plot(x, s)
plt.title('ReLU')
plt.axvline(0);

### Tangente hiperbólica (tanh)

A tanh também tem a forma de um "S", mas leva os valores para o intervalo $(-1, 1)$ e é centrada em zero:

$$\tanh(x) = \frac{e^{x} - e^{-x}}{e^{x} + e^{-x}} = \frac{2}{1+e^{-2x}} - 1 = 2\,\sigma(2x) - 1$$

A última igualdade mostra que a tanh é uma sigmoide esticada e deslocada. Sua derivada, $1 - \tanh^2(x)$, vale 1 em $x = 0$ e também tende a zero quando $|x|$ é grande.

In [ ]:
x = np.linspace(-4, 4, 100)

s = np.tanh(x)

plt.plot(x, s)
plt.title('Tanh')
plt.axhline(0, ls='dashed')
plt.axvline(0);

### Softsign

A softsign tem forma parecida com a tanh e o mesmo intervalo $(-1, 1)$, mas se aproxima das assíntotas mais devagar (como $1/|x|$, e não exponencialmente):

$$f(x) = \frac{x}{1 + |x|}$$

In [ ]:
x = np.linspace(-8, 8, 100)

s = x / (1 + np.abs(x))

plt.plot(x, s)
plt.title('Softsign')
plt.axhline(0, ls='dashed')
plt.axvline(0);

### ELU (Exponential Linear Unit)

A ELU é igual à ReLU para $x \ge 0$, mas para $x < 0$ tem uma curva suave que tende a $-\alpha$:

$$f(x) = \begin{cases} x & \text{se } x \ge 0 \\ \alpha\,(e^{x} - 1) & \text{caso contrário} \end{cases}$$

$\alpha > 0$ é um hiperparâmetro (o padrão do `nn.ELU` é $\alpha = 1$). Com $\alpha = 0$, a ELU vira a ReLU. Como a saída pode ser negativa, a média das ativações fica mais perto de zero, o que, segundo Clevert et al. (2015), acelera o aprendizado.

In [ ]:
alpha = 1.0

x = np.linspace(-8, 8, 100)

s = alpha * (np.exp(x) - 1)
s[x > 0] = x[x > 0]

plt.plot(x, s)
plt.title('ELU: Exponential Linear Unit')
plt.axhline(0, ls='dashed')
plt.axvline(0);

## Ativações no PyTorch

No PyTorch, as ativações são classes do `torch.nn` derivadas de `nn.Module`, como o `nn.Linear`: primeiro criamos o objeto (`relu = nn.ReLU()`) e depois o chamamos sobre um tensor (`relu(x)`), como se fosse uma função. Assim, elas podem ser colocadas entre as camadas de um `nn.Sequential`, como no 4.2. Também existem versões em forma de função, como `torch.relu(x)` e `torch.sigmoid(x)`.

As ativações recebem tensores comuns. Versões antigas do PyTorch exigiam embrulhar o tensor em `torch.autograd.Variable`; isso é obsoleto desde o PyTorch 0.4 e não é mais necessário.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0);  # semente fixa para reprodutibilidade

### ReLU

In [ ]:
relu = nn.ReLU()                   # criação do objeto da ativação
x = torch.linspace(-8, 8, 100)     # tensor comum, shape (100,)
s = relu(x)                        # aplica a ReLU elemento a elemento

plt.plot(x.numpy(), s.numpy())     # .numpy() converte o tensor para array NumPy
plt.title('ReLU: Rectified Linear Unit')
plt.axhline(0, ls='dashed')
plt.axvline(0);

### Várias ativações do PyTorch e suas derivadas

Além do valor da ativação, vamos desenhar a sua derivada $f'(x)$, calculada pelo autograd (visto no 2.4):

- `x` é criado com `requires_grad=True`, para que o PyTorch registre as operações feitas sobre ele;
- `s.sum().backward()` calcula o gradiente de $\sum_i f(x_i)$ em relação a cada $x_i$. Como cada saída $f(x_i)$ depende só da sua própria entrada $x_i$, esse gradiente é exatamente $f'(x_i)$, e fica guardado em `x.grad`;
- `.detach()` devolve o tensor sem o registro do gradiente, necessário antes de `.numpy()`.

A derivada importa porque, na retropropagação (backpropagation), o gradiente que chega a uma camada é multiplicado pela derivada da ativação. Onde $f'(x) \approx 0$ (a ativação **satura**), o gradiente praticamente some e os pesos anteriores quase não são atualizados.

In [ ]:
def plot_activations(activations, x_min=-8, x_max=8):
    '''Desenha cada ativação (linha cheia) e sua derivada calculada pelo autograd (linha tracejada).'''
    if not activations:
        print('Nenhuma ativação para desenhar.')
        return
    n_rows = (len(activations) + 1) // 2
    fig = plt.figure(figsize=(12, 3.5 * n_rows))
    for k, (title, activation) in enumerate(activations.items()):
        m = activation()                                         # cria o objeto da ativação
        x = torch.linspace(x_min, x_max, 200, requires_grad=True)
        s = m(x)
        s.sum().backward()                                       # x.grad passa a conter f'(x)
        ax = fig.add_subplot(n_rows, 2, k + 1)
        ax.plot(x.detach().numpy(), s.detach().numpy(), label='f(x)')
        ax.plot(x.detach().numpy(), x.grad.numpy(), '--', label="f'(x)")
        ax.set_title(title)
        ax.axhline(0, color='gray', lw=0.8)
        ax.axvline(0, color='gray', lw=0.8)
        ax.grid(True)
        ax.legend()
    fig.tight_layout()
    plt.show()


activations = {'ReLU: Rectified Linear Unit': nn.ReLU,
               'ReLU6: ReLU limitada em 6': nn.ReLU6,
               'ELU: Exponential Linear Unit': nn.ELU,
               'SELU: Scaled Exponential Linear Unit': nn.SELU,
               'Sigmoid': nn.Sigmoid,
               'Tanh': nn.Tanh,
               }
plot_activations(activations)

O que observar nos gráficos:

- **Sigmoid** e **Tanh** saturam dos dois lados: para $|x| > 4$, a derivada é quase zero. A derivada da sigmoide nunca passa de $0{,}25$.
- **ReLU** tem derivada 1 para $x > 0$ e 0 para $x < 0$. A **ReLU6** é a ReLU limitada em 6 ($\min(\max(0, x), 6)$), e por isso tem derivada zero também para $x > 6$.
- **ELU** tem derivada contínua em $x = 0$ (com $\alpha = 1$) e saída negativa para $x < 0$.
- **SELU** é a ELU multiplicada por uma constante, $\text{SELU}(x) = \lambda\,\text{ELU}_\alpha(x)$ com $\lambda \approx 1{,}0507$ e $\alpha \approx 1{,}6733$, valores escolhidos por Klambauer et al. (2017) para que as ativações mantenham média zero e variância um ao longo das camadas.

## Exercícios

**1.** Escolha pelo menos quatro ativações do `torch.nn` que não estão na lista acima (sugestões: `nn.LeakyReLU`, `nn.GELU`, `nn.SiLU`, `nn.Softplus`, `nn.Softsign`, `nn.Hardtanh`, `nn.Mish`) e desenhe-as com `plot_activations`. A lista completa está na [documentação do PyTorch](https://docs.pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity). Para cada uma, diga se ela satura e de que lado.

In [ ]:
more_activations = {
    # TODO: escreva sua solução aqui
}
plot_activations(more_activations)

*Escreva sua resposta aqui.*

**2.** Confira as implementações em NumPy do início do notebook: calcule a sigmoide, a tanh, a softsign e a ELU ($\alpha = 1$) com NumPy e com o PyTorch (`torch.sigmoid`, `torch.tanh`, `nn.Softsign`, `nn.ELU`) para os mesmos valores de `x_np` e mostre a maior diferença absoluta entre elas.

In [ ]:
x_np = np.linspace(-8, 8, 100)
x_t = torch.from_numpy(x_np)    # mesmo dado, como tensor (float64)

# TODO: escreva sua solução aqui


**3.** Duas camadas lineares sem ativação entre elas equivalem a uma única camada linear. Verifique isso numericamente: crie `lin1 = nn.Linear(2, 5)` e `lin2 = nn.Linear(5, 3)`, calcule (sem gradiente, dentro de `with torch.no_grad():`) $W = W_2W_1$ e $\boldsymbol{b} = W_2\boldsymbol{b}_1 + \boldsymbol{b}_2$ a partir de `lin1.weight`, `lin1.bias`, `lin2.weight` e `lin2.bias`, e compare `lin2(lin1(x))` com `x @ W.T + b` para um lote `x` de shape (10, 2). Depois coloque uma ReLU entre as duas camadas e compare de novo. O que a diferença mostra?

In [ ]:
torch.manual_seed(0)
lin1 = nn.Linear(2, 5)
lin2 = nn.Linear(5, 3)
x = torch.randn(10, 2)

# TODO: escreva sua solução aqui


*Escreva sua resposta aqui.*

## Aprendizados

- Sem uma ativação não linear entre as camadas, uma rede com várias camadas equivale a um único modelo linear.
- Sigmoide e tanh saturam: para $|x|$ grande, a derivada é quase zero e o gradiente diminui ao passar por elas na retropropagação. A ReLU tem derivada 1 para $x > 0$ e é a escolha padrão nas camadas escondidas.
- No PyTorch, as ativações são módulos (`nn.ReLU()`, `nn.Sigmoid()`, ...) aplicados a tensores comuns; `torch.autograd.Variable` não é mais necessário.
- Com `requires_grad=True` e `backward()`, o autograd calcula a derivada de qualquer ativação, sem derivá-la à mão.
- A GELU do exercício 1 é a ativação das camadas feed-forward do BERT e do GPT-2.

## Referências

- [Activation function (Wikipedia)](https://en.wikipedia.org/wiki/Activation_function) — quadro geral de várias ativações, com equações, gráficos e derivadas.
- [Rectifier (neural networks) (Wikipedia)](https://en.wikipedia.org/wiki/Rectifier_(neural_networks)) — ativações baseadas em retificação (ReLU e variantes).
- [PyTorch: funções de ativação (torch.nn)](https://docs.pytorch.org/docs/stable/nn.html#non-linear-activations-weighted-sum-nonlinearity)
- [Glorot, X.; Bordes, A.; Bengio, Y. (2011). Deep Sparse Rectifier Neural Networks. AISTATS.](https://proceedings.mlr.press/v15/glorot11a/glorot11a.pdf)
- [Clevert, D.-A.; Unterthiner, T.; Hochreiter, S. (2015). Fast and Accurate Deep Network Learning by Exponential Linear Units (ELUs).](https://arxiv.org/abs/1511.07289)
- [Klambauer, G. et al. (2017). Self-Normalizing Neural Networks (SELU).](https://arxiv.org/abs/1706.02515)
- [Hendrycks, D.; Gimpel, K. (2016). Gaussian Error Linear Units (GELUs).](https://arxiv.org/abs/1606.08415)
- [Devlin, J. et al. (2018). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.](https://arxiv.org/abs/1810.04805)